In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%PandasProfile
### cell 11 ###

use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"
# work on a pandas DataFrame when using Modin, otherwise use df directly
df_pd = df._to_pandas() if use_modin else df

# top 11 countries by count
country_order = df_pd['first_country'].value_counts().index[:11]

# faster count of type per country
counts = pd.crosstab(df_pd['first_country'], df_pd['type']).loc[country_order]

# wrap back into a Modin DataFrame if needed
if use_modin:
    data_q2q3 = pd.DataFrame(counts)
else:
    data_q2q3 = counts

# preserve the "sum" column as before
data_q2q3['sum'] = data_q2q3.sum(axis=1)

# compute ratios without transposes and sort exactly as original
data_q2q3_ratio = (
    data_q2q3
      .div(data_q2q3['sum'], axis=0)[['Movie', 'TV Show']]
      .sort_values(by='Movie', ascending=False)
      .iloc[::-1]
)